# VQA CAA

## Gemma 3

### defs

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import json
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm
import kagglehub
from pathlib import Path

In [ ]:
model_id = "google/gemma-3-4b-it"

In [ ]:
def load_model():
  processor = AutoProcessor.from_pretrained(model_id)
  model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",            # Automatically loads model onto available GPU(s)
    dtype=torch.bfloat16    # Use bfloat16 for optimized memory and inference
  ).eval()
  return processor, model

In [ ]:
def make_q(img_pth, q_txt):
  image = Image.open(img_pth).convert("RGB")
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": q_txt}
        ]
    }
  ]
  return messages

In [ ]:
def generate_a(messages, processor, model):
  inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
  ).to(model.device)

  with torch.inference_mode():
    outputs = model(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Set to True if you want more creative/varied responses
    )
  generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
  decoded_output = processor.decode(generated_ids, skip_special_tokens=True)

  return decoded_output

In [ ]:
proc, mod = load_model()

In [ ]:
mod

In [ ]:
def print_answ(img_pth, q_txt):
  print(generate_a(make_q(img_pth, q_txt), proc, mod))

### examples

In [ ]:
m = 'Is there anything else that is the same shape as the small brown matte object?'
img = "/content/CLEVR_test_000000.png"
print_answ(img, m)

In [ ]:
print_answ(img, m)

## CAA vect application

### DS

In [ ]:
path = kagglehub.dataset_download("timoboz/clevr-dataset")
print("Path to dataset files:", path)

In [ ]:
class CLEVRCAAEvaluationDataset(Dataset):
    def __init__(self, jsonl_path, image_dir):
        print(f"Loading official benchmark from {jsonl_path}...")
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        self.questions = data['questions']
        self.image_dir = Path(image_dir)
        print(f"Successfully loaded {len(self.questions)} validation questions.")

    def __len__(self): return len(self.questions)

    def __getitem__(self, idx):
        q = self.questions[idx]
        image_path = self.image_dir / q['image_filename']
        image = Image.open(image_path).convert('RGB')
        true_answer = str(q['answer'])
        q_text = q['question']
        
        # Tag the question type so we can test for Concept Leakage later
        is_counting = true_answer.isdigit() # and q_text.lower().startswith("how many")
        q_type = "count" if is_counting else "other"
        
        return {
            "image": image,
            "question": q_text,
            "true_answer": true_answer,
            "q_type": q_type
        }

In [ ]:
def pil_collate_fn(batch):
    """
    Custom collate function to handle PIL Images.
    Takes a list of dictionaries (the batch) and groups them by key.
    """
    return {key: [item[key] for item in batch] for key in batch[0].keys()}

### 2. Identifying the CLEVR Token IDs

In [ ]:
valid_clevr_answers = [str(i) for i in range(10)] # '0' through '9'

allowed_token_ids = []
token_to_string_map = {}

print("Mapping CLEVR answers to Gemma Token IDs:")
for ans in valid_clevr_answers:
    # We encode just the number.
    # add_special_tokens=False prevents adding a <bos> token.
    token_ids = proc.tokenizer.encode(ans, add_special_tokens=False)

    assert len(token_ids) == 1, f"Error: '{ans}' split into multiple tokens!"
    token_id = token_ids[0]

    allowed_token_ids.append(token_id)
    token_to_string_map[token_id] = ans

    print(f"Answer: '{ans}' -> Token ID: {token_id}")

### Hook

In [ ]:
class SteeringHook:
    def __init__(self, model, steering_vector, target_layer, coefficient):
        self.model = model
        self.steering_vector = steering_vector
        self.target_layer = target_layer
        self.coefficient = coefficient
        self.handle = None

    def _hook_fn(self, module, input, output):
        # 1. Safely locate the hidden states
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output

        # 2. Reshape and move vector to guarantee mathematical alignment
        # steering_vector is (hidden_size,) -> view as (1, 1, hidden_size)
        sv = self.steering_vector.to(hidden_states.device, dtype=hidden_states.dtype)
        if sv.dim() == 1:
            sv = sv.view(1, 1, -1)

        # 3. Modify IN-PLACE!
        # This completely bypasses PyTorch's tuple-returning mechanics,
        # guaranteeing we don't break the model's forward pass.
        hidden_states.add_(self.coefficient * sv)

    def attach(self):
        layer_module = self.model.model.language_model.layers[self.target_layer]
        self.handle = layer_module.register_forward_hook(self._hook_fn)

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

### 3. The Upgraded Evaluation Loop

In [ ]:
def get_masked_clevr_prediction(logits, allowed_token_ids, token_to_string_map):
    """Filters the 256k vocab logits down to just the CLEVR valid answers."""
    vocab_size = logits.shape[-1]

    # --- THE FIX: Explicitly set dtype=logits.dtype so it matches Gemma's BFloat16 ---
    mask = torch.full(
        (1, vocab_size),
        float('-inf'),
        device=logits.device,
        dtype=logits.dtype
    )

    allowed_indices = torch.tensor(allowed_token_ids, device=logits.device)
    mask[0, allowed_indices] = logits[0, allowed_indices]

    # Best practice: Upcast to float32 just for the softmax calculation
    # to prevent numerical overflow/underflow, then calculate probabilities.
    probabilities = F.softmax(mask.to(torch.float32), dim=-1)

    predicted_token_id = torch.argmax(probabilities, dim=-1).item()

    prob_dict = {
        token_to_string_map[tid]: probabilities[0, tid].item()
        for tid in allowed_token_ids
    }

    return token_to_string_map[predicted_token_id], prob_dict

In [ ]:
def evaluate_steering_vector(model, processor, dataloader, steering_vector, target_layer, multiplier, allowed_token_ids, token_map):
    # 1. Attach the Steering Hook
    hook = SteeringHook(model, steering_vector, target_layer, multiplier)
    hook.attach()
    model.eval()

    metrics = defaultdict(lambda: {"total": 0, "correct_base": 0, "correct_shifted": 0, "p_true": 0.0, "p_target": 0.0})

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval Multiplier: {multiplier}"):
            # 1. Grab the raw data from your batch
            q = batch["question"][0]
            image = batch["image"][0]
            true_ans = batch["true_answer"][0]
            target_ans = batch["target_answer"][0]
            q_type = batch["q_type"][0]

            # 2. Build the EXACT same Multimodal Message Array
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": q}
                    ]
                }
            ]

            # 3. Apply the Official Template
            eval_prompt = processor.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True 
            )

            # 3. Forward Pass (Vector is injected automatically by the hook!)
            inputs = processor(text=eval_prompt, images=image, return_tensors="pt").to(model.device)
            outputs = model(**inputs)

            # Get logits for the VERY NEXT token the model wants to predict
            next_token_logits = outputs.logits[:, -1, :]

            # 4. Mask and Predict
            pred_ans, prob_dict = get_masked_clevr_prediction(next_token_logits, allowed_token_ids, token_map)

            # 5. Record Metrics
            stats = metrics[q_type]
            stats["total"] += 1
            stats["p_true"] += prob_dict.get(true_ans, 0.0)
            stats["p_target"] += prob_dict.get(target_ans, 0.0)

            if pred_ans == true_ans:
                stats["correct_base"] += 1
            if pred_ans == target_ans:
                stats["correct_shifted"] += 1

    # 6. Crucial: Remove the hook so it doesn't corrupt future runs
    hook.remove()
    return metrics

### 4. Running the Multiplier Sweep

In [ ]:
eval_dataset = CLEVRCAAEvaluationDataset('/content/clevr_caa_extract_val.jsonl', path)
eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

In [ ]:
multipliers_to_test = [0.0, 0.5, 1.0, 1.5, 2.0]

In [ ]:
for target_layer in range(16):
    print(f"\n--- Running Evaluation on layer: {target_layer} ---")
    steering_vector = torch.load(f"/content/clevr_overcount_plus1_layer{target_layer}.pt")
    sweep_results = {}

    for mult in multipliers_to_test:
        print(f"\n--- Running Evaluation with Multiplier: {mult} ---")
        metrics = evaluate_steering_vector(
            model=mod,
            processor=proc,
            dataloader=eval_loader,
            steering_vector=steering_vector,
            target_layer=target_layer,
            multiplier=mult,
            allowed_token_ids=allowed_token_ids,
            token_map=token_to_string_map
        )
        sweep_results[mult] = metrics

    print("\n=== FINAL SWEEP RESULTS ===")
    for mult, metrics in sweep_results.items():
        print(f"\nMultiplier: {mult}")
        for q_type, stats in metrics.items():
            total = stats["total"]
            if total == 0: continue

            base_acc = (stats["correct_base"] / total) * 100
            shift_acc = (stats["correct_shifted"] / total) * 100

            print(f"  Type: {q_type.ljust(15)} | Base Acc: {base_acc:5.1f}% | Shifted Acc: {shift_acc:5.1f}% | Avg P(Target): {stats['p_target']:5.3f}")            